# Exploring Artwork Embeddings

This notebook explores the embeddings generated for artwork identification.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'backend'))

import numpy as np
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# Load the index
index_path = Path("../backend/artworks_index.pkl")
with open(index_path, 'rb') as f:
    index_data = pickle.load(f)

embeddings = index_data["embeddings"]
metadata = index_data["metadata"]

print(f"Loaded {len(metadata)} artworks")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Embedding shape: {embeddings.shape}")


## Basic Statistics


In [ ]:
# Compute basic statistics
print("Embedding Statistics:")
print(f"Mean: {embeddings.mean():.4f}")
print(f"Std: {embeddings.std():.4f}")
print(f"Min: {embeddings.min():.4f}")
print(f"Max: {embeddings.max():.4f}")

# Check for NaN or Inf values
print(f"\nNaN values: {np.isnan(embeddings).sum()}")
print(f"Inf values: {np.isinf(embeddings).sum()}")


## Visualize Embedding Distribution


In [ ]:
# Sample a subset for visualization
sample_size = min(1000, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
sample_embeddings = embeddings[sample_indices]

# Flatten for histogram
plt.figure(figsize=(10, 6))
plt.hist(sample_embeddings.flatten(), bins=50, alpha=0.7)
plt.title("Distribution of Embedding Values")
plt.xlabel("Embedding Value")
plt.ylabel("Frequency")
plt.show()


## Dimensionality Reduction Visualization


In [ ]:
# Use PCA for faster visualization
print("Computing PCA...")
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(sample_embeddings)

# Color by artist (if available)
artists = [m.get('artist', 'Unknown') for m in metadata]
sample_artists = [artists[i] for i in sample_indices]

plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                     c=range(len(sample_embeddings)), 
                     cmap='viridis', alpha=0.6, s=20)
plt.colorbar(scatter)
plt.title("PCA Visualization of Artwork Embeddings")
plt.xlabel(f"PC1 (explained variance: {pca.explained_variance_ratio_[0]:.2%})")
plt.ylabel(f"PC2 (explained variance: {pca.explained_variance_ratio_[1]:.2%})")
plt.show()

print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2%}")


## Similarity Analysis


In [ ]:
# Compute pairwise similarities for a sample
from sklearn.metrics.pairwise import cosine_similarity

sample_size_sim = min(100, len(embeddings))
sample_indices_sim = np.random.choice(len(embeddings), sample_size_sim, replace=False)
sample_emb_sim = embeddings[sample_indices_sim]

# Normalize for cosine similarity
sample_emb_norm = sample_emb_sim / (np.linalg.norm(sample_emb_sim, axis=1, keepdims=True) + 1e-8)
similarities = cosine_similarity(sample_emb_norm)

# Remove diagonal (self-similarity)
np.fill_diagonal(similarities, 0)

plt.figure(figsize=(10, 8))
sns.heatmap(similarities, cmap='viridis', cbar=True)
plt.title("Pairwise Cosine Similarity Matrix (Sample)")
plt.show()

# Distribution of similarities
similarities_flat = similarities[similarities != 0]
plt.figure(figsize=(10, 6))
plt.hist(similarities_flat, bins=50, alpha=0.7)
plt.title("Distribution of Pairwise Similarities")
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.show()

print(f"Mean similarity: {similarities_flat.mean():.4f}")
print(f"Max similarity: {similarities_flat.max():.4f}")
print(f"Min similarity: {similarities_flat.min():.4f}")


## Test Search Functionality


In [ ]:
from search import ArtworkSearcher

# Initialize searcher
searcher = ArtworkSearcher("../backend/artworks_index.pkl")

# Test text search
test_queries = [
    "starry night",
    "portrait of a woman",
    "abstract painting",
    "landscape with mountains"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = searcher.search_by_text(query, top_k=3)
    for i, result in enumerate(results, 1):
        print(f"  {i}. {result.get('title', 'Unknown')} by {result.get('artist', 'Unknown')} "
              f"(score: {result.get('similarity_score', 0):.4f})")
